# MCNEMAR

In [4]:
"""
mcnemar_test.py
================

Modul generik dan reusable untuk melakukan McNemar Test terhadap dua
metode klasifikasi apa pun, berdasarkan file Excel dengan struktur sheet
"Detail Comparison" yang berisi kolom `ground_truth` dan tepat dua kolom
prediksi berakhiran `_prediction`.

Tidak ada nama metode yang di-hardcode: nama metode diambil otomatis dari
nama kolom `_prediction` yang ditemukan di dalam file.

Modul ini dirancang untuk dipakai sepenuhnya dari dalam Jupyter Notebook /
Google Colab (tidak ada antarmuka command-line).

Cara pakai di dalam cell notebook:

    from mcnemar_test import analyze

    result, output_path = analyze("data.xlsx")

    # hasil bisa langsung diinspeksi tanpa membuka file Excel:
    result.df.head()          # DataFrame asli + kolom *_correct
    result.p_value
    result.statistic
    result.decision
    result.accuracy_method1
    result.accuracy_method2
    print(output_path)        # path file Excel <nama_file>_mcnemar.xlsx

Parameter opsional (semua punya default yang sesuai spesifikasi):

    result, output_path = analyze(
        "data.xlsx",
        sheet_name="Detail Comparison",   # default
        alpha=0.05,                        # default
        output_path=None,                  # default -> <nama_file>_mcnemar.xlsx
        verbose=True,                       # default -> cetak ringkasan di cell
    )

Requirement:
    pandas, statsmodels, openpyxl
"""

from __future__ import annotations

import sys
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Optional
import math
import pandas as pd
import statsmodels
from statsmodels.stats.contingency_tables import mcnemar

from openpyxl import Workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.worksheet.worksheet import Worksheet


SHEET_SOURCE_NAME = "Detail Comparison"
PREDICTION_SUFFIX = "_prediction"
GROUND_TRUTH_COL = "ground_truth"


# --------------------------------------------------------------------------- #
# Exceptions
# --------------------------------------------------------------------------- #

class McNemarAnalysisError(Exception):
    """Exception dasar untuk semua error terkait analisis McNemar."""


class FileNotFoundInAnalysisError(McNemarAnalysisError):
    """File Excel input tidak ditemukan."""


class SheetNotFoundError(McNemarAnalysisError):
    """Sheet yang dibutuhkan tidak ditemukan di dalam file Excel."""


class MissingColumnError(McNemarAnalysisError):
    """Kolom wajib (ground_truth) tidak ditemukan."""


class InvalidPredictionColumnCountError(McNemarAnalysisError):
    """Jumlah kolom berakhiran _prediction bukan tepat dua."""


class EmptyDataError(McNemarAnalysisError):
    """Data kosong / tidak ada baris valid untuk dianalisis."""


# --------------------------------------------------------------------------- #
# Struktur data hasil analisis
# --------------------------------------------------------------------------- #

@dataclass
class McNemarResult:
    """Menyimpan seluruh hasil perhitungan McNemar Test."""

    method1_name: str
    method2_name: str
    df: pd.DataFrame                 # data asli + kolom *_correct
    total_data: int
    accuracy_method1: float
    accuracy_method2: float
    a: int
    b: int
    c: int
    d: int
    is_exact: bool
    use_correction: bool
    statistic: float
    p_value: float
    alpha: float
    reject_null: bool = field(init=False)
    decision: str = field(init=False)
    interpretation: str = field(init=False)
    effect_size: EffectSizeResult | None = None

    def __post_init__(self) -> None:
        self.reject_null = self.p_value < self.alpha
        self.decision = "Reject H0" if self.reject_null else "Fail to Reject H0"
        if self.reject_null:
            self.interpretation = (
                f"p-value ({self.p_value:.6f}) < alpha ({self.alpha}) sehingga H0 ditolak. "
                f"Ini menunjukkan bahwa kedua metode ('{self.method1_name}' dan "
                f"'{self.method2_name}') memiliki performa yang berbeda secara signifikan."
            )
        else:
            self.interpretation = (
                f"p-value ({self.p_value:.6f}) >= alpha ({self.alpha}) sehingga gagal "
                f"menolak H0. Ini menunjukkan tidak terdapat perbedaan performa yang "
                f"signifikan antara metode '{self.method1_name}' dan '{self.method2_name}'."
            )

@dataclass
class EffectSizeResult:
    """Menyimpan hasil effect size McNemar."""

    odds_ratio: float
    log_odds_ratio: float
    standard_error: float
    ci_lower: float
    ci_upper: float
    direction: str
    magnitude: str
    interpretation: str

# --------------------------------------------------------------------------- #
# 1. Membaca & memvalidasi file Excel
# --------------------------------------------------------------------------- #

def load_detail_comparison(
    file_path: Path,
    sheet_name: str = SHEET_SOURCE_NAME,
) -> pd.DataFrame:
    """
    Membaca sheet "Detail Comparison" dari file Excel.

    Raises:
        FileNotFoundInAnalysisError: jika file tidak ditemukan.
        SheetNotFoundError: jika sheet tidak ditemukan.
        EmptyDataError: jika sheet kosong.
    """
    if not file_path.exists():
        raise FileNotFoundInAnalysisError(f"File tidak ditemukan: '{file_path}'")

    if not file_path.is_file():
        raise FileNotFoundInAnalysisError(f"Path bukan file yang valid: '{file_path}'")

    try:
        excel_file = pd.ExcelFile(file_path)
    except Exception as exc:  # noqa: BLE001
        raise McNemarAnalysisError(
            f"Gagal membuka file Excel '{file_path}': {exc}"
        ) from exc

    if sheet_name not in excel_file.sheet_names:
        raise SheetNotFoundError(
            f"Sheet '{sheet_name}' tidak ditemukan di dalam file '{file_path}'. "
            f"Sheet yang tersedia: {excel_file.sheet_names}"
        )

    df = excel_file.parse(sheet_name=sheet_name)

    if df.empty:
        raise EmptyDataError(
            f"Sheet '{sheet_name}' pada file '{file_path}' tidak berisi data."
        )

    return df


def detect_prediction_columns(df: pd.DataFrame) -> tuple[str, str]:
    """
    Mencari kolom yang berakhiran `_prediction` secara otomatis.

    Returns:
        Tuple (kolom_prediksi_1, kolom_prediksi_2) sesuai urutan kemunculan
        di dalam dataframe.

    Raises:
        InvalidPredictionColumnCountError: jika jumlah kolom bukan tepat dua.
    """
    prediction_cols = [c for c in df.columns if str(c).endswith(PREDICTION_SUFFIX)]

    if len(prediction_cols) != 2:
        raise InvalidPredictionColumnCountError(
            f"Dibutuhkan tepat 2 kolom yang berakhiran '{PREDICTION_SUFFIX}', "
            f"namun ditemukan {len(prediction_cols)}: {prediction_cols}"
        )

    return prediction_cols[0], prediction_cols[1]


def validate_ground_truth_column(df: pd.DataFrame) -> None:
    """
    Memastikan kolom ground_truth ada di dalam dataframe.

    Raises:
        MissingColumnError: jika kolom ground_truth tidak ditemukan.
    """
    if GROUND_TRUTH_COL not in df.columns:
        raise MissingColumnError(
            f"Kolom wajib '{GROUND_TRUTH_COL}' tidak ditemukan di dalam data. "
            f"Kolom yang tersedia: {list(df.columns)}"
        )


def extract_method_name(prediction_column: str) -> str:
    """Menghapus suffix `_prediction` untuk mendapatkan nama metode."""
    return prediction_column[: -len(PREDICTION_SUFFIX)]


def validate_no_missing_values(
    df: pd.DataFrame, columns: list[str]
) -> None:
    """
    Memastikan kolom-kolom yang dibutuhkan tidak seluruhnya kosong dan
    tidak mengandung baris yang benar-benar tidak dapat dianalisis.

    Raises:
        EmptyDataError: jika setelah dropna tidak ada baris tersisa.
    """
    subset = df[columns]
    if subset.dropna(how="any").empty:
        raise EmptyDataError(
            "Tidak ada baris data yang valid (semua baris memiliki nilai kosong "
            f"pada salah satu kolom: {columns})."
        )


# --------------------------------------------------------------------------- #
# 2. Membentuk kolom correct & tabel kontingensi
# --------------------------------------------------------------------------- #

def build_correctness_columns(
    df: pd.DataFrame,
    pred_col1: str,
    pred_col2: str,
    method1_name: str,
    method2_name: str,
) -> pd.DataFrame:
    """
    Menambahkan kolom `<Method>_correct` (boolean) berdasarkan
    prediction == ground_truth. Mengembalikan salinan dataframe baru.
    """
    result_df = df.copy()

    gt = result_df[GROUND_TRUTH_COL]
    result_df[f"{method1_name}_correct"] = result_df[pred_col1] == gt
    result_df[f"{method2_name}_correct"] = result_df[pred_col2] == gt

    return result_df


def compute_accuracy(df: pd.DataFrame, correct_col: str) -> float:
    """Menghitung accuracy (proporsi True) dari sebuah kolom boolean correct."""
    return float(df[correct_col].mean())


def build_contingency_counts(
    df: pd.DataFrame,
    method1_correct_col: str,
    method2_correct_col: str,
) -> tuple[int, int, int, int]:
    """
    Membentuk tabel kontingensi McNemar 2x2.

    Returns:
        a = kedua metode benar
        b = Method1 benar, Method2 salah
        c = Method1 salah, Method2 benar
        d = kedua metode salah
    """
    m1 = df[method1_correct_col]
    m2 = df[method2_correct_col]

    a = int(((m1) & (m2)).sum())
    b = int(((m1) & (~m2)).sum())
    c = int(((~m1) & (m2)).sum())
    d = int(((~m1) & (~m2)).sum())

    return a, b, c, d


# --------------------------------------------------------------------------- #
# 3. Menjalankan McNemar Test
# --------------------------------------------------------------------------- #

def run_mcnemar_statistical_test(
    a: int, b: int, c: int, d: int
) -> tuple[bool, bool, float, float]:
    """
    Menjalankan McNemar Test menggunakan statsmodels.

    Aturan penentuan jenis test:
        if (b + c) < 25: exact = True
        else: exact = False; correction = True

    Returns:
        (is_exact, use_correction, statistic, p_value)
    """
    table = [[a, b], [c, d]]

    b_plus_c = b + c
    if b_plus_c < 25:
        is_exact = True
        use_correction = False  # continuity correction tidak relevan untuk exact test
    else:
        is_exact = False
        use_correction = True

    test_result = mcnemar(table, exact=is_exact, correction=use_correction)

    return is_exact, use_correction, float(test_result.statistic), float(test_result.pvalue)

def compute_effect_size(
    b: int,
    c: int,
    method1_name: str,
    method2_name: str,
) -> EffectSizeResult:
    """
    Menghitung Matched-pairs Odds Ratio beserta 95% Confidence Interval.
    OR = c / b
    Parameters
    ----------
    b : int
        Method1 benar, Method2 salah
    c : int
        Method1 salah, Method2 benar
    """

    # Haldane-Anscombe correction
    # menghindari pembagian dengan nol
    b_adj = b if b > 0 else 0.5
    c_adj = c if c > 0 else 0.5

    odds_ratio = c_adj / b_adj

    log_or = math.log(odds_ratio)

    se = math.sqrt((1 / b_adj) + (1 / c_adj))

    ci_lower = math.exp(log_or - 1.96 * se)
    ci_upper = math.exp(log_or + 1.96 * se)

    # arah perbedaan
    if odds_ratio > 1:
        direction = f"{method2_name} Better"
    elif odds_ratio < 1:
        direction = f"{method1_name} Better"
    else:
        direction = "Equal"

    # klasifikasi besar efek
    if 0.90 <= odds_ratio <= 1.10:
        magnitude = "Negligible"
    elif odds_ratio < 1:
        magnitude = "Small"
    elif odds_ratio <= 1.5:
        magnitude = "Small"
    elif odds_ratio <= 3:
        magnitude = "Moderate"
    else:
        magnitude = "Large"

    if odds_ratio > 1:
        interpretation = (
            f"{method2_name} sekitar {odds_ratio:.2f} kali lebih sering "
            f"memperbaiki kesalahan dibandingkan {method1_name}."
        )

    elif odds_ratio < 1:
        interpretation = (
            f"{method1_name} sekitar {1/odds_ratio:.2f} kali lebih sering "
            f"memperbaiki kesalahan dibandingkan {method2_name}."
        )

    else:
        interpretation = (
            "Kedua metode memiliki peluang perbaikan yang sama."
        )

    return EffectSizeResult(
        odds_ratio=odds_ratio,
        log_odds_ratio=log_or,
        standard_error=se,
        ci_lower=ci_lower,
        ci_upper=ci_upper,
        direction=direction,
        magnitude=magnitude,
        interpretation=interpretation,
    )

# --------------------------------------------------------------------------- #
# 4. Orkestrasi analisis lengkap
# --------------------------------------------------------------------------- #

def analyze_mcnemar(
    file_path: Path,
    sheet_name: str = SHEET_SOURCE_NAME,
    alpha: float = 0.05,
) -> McNemarResult:
    """
    Menjalankan seluruh pipeline analisis McNemar dari file Excel input,
    dan mengembalikan objek McNemarResult.
    """
    df = load_detail_comparison(file_path, sheet_name=sheet_name)

    validate_ground_truth_column(df)
    pred_col1, pred_col2 = detect_prediction_columns(df)

    validate_no_missing_values(df, [GROUND_TRUTH_COL, pred_col1, pred_col2])

    method1_name = extract_method_name(pred_col1)
    method2_name = extract_method_name(pred_col2)

    enriched_df = build_correctness_columns(
        df, pred_col1, pred_col2, method1_name, method2_name
    )

    method1_correct_col = f"{method1_name}_correct"
    method2_correct_col = f"{method2_name}_correct"

    accuracy1 = compute_accuracy(enriched_df, method1_correct_col)
    accuracy2 = compute_accuracy(enriched_df, method2_correct_col)

    a, b, c, d = build_contingency_counts(
        enriched_df, method1_correct_col, method2_correct_col
    )

    is_exact, use_correction, statistic, p_value = run_mcnemar_statistical_test(a, b, c, d)
    effect_size = compute_effect_size(
        b,
        c,
        method1_name,
        method2_name,
    )

    return McNemarResult(
        method1_name=method1_name,
        method2_name=method2_name,
        df=enriched_df,
        total_data=len(enriched_df),
        accuracy_method1=accuracy1,
        accuracy_method2=accuracy2,
        a=a,
        b=b,
        c=c,
        d=d,
        is_exact=is_exact,
        use_correction=use_correction,
        statistic=statistic,
        p_value=p_value,
        alpha=alpha,
        effect_size=effect_size,
    )


# --------------------------------------------------------------------------- #
# 5. Penulisan hasil ke Excel (openpyxl) dengan styling
# --------------------------------------------------------------------------- #

HEADER_FILL = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
HEADER_FONT = Font(bold=True, color="FFFFFF")
TITLE_FONT = Font(bold=True, size=12)
LABEL_FONT = Font(bold=True)
THIN_BORDER = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"), bottom=Side(style="thin"),
)
CENTER_ALIGN = Alignment(horizontal="center", vertical="center")


def _autofit_columns(ws: Worksheet, min_width: int = 10, max_width: int = 60) -> None:
    """Mengatur lebar kolom secara otomatis berdasarkan panjang konten terpanjang."""
    for col_cells in ws.columns:
        length = 0
        col_letter = None
        for cell in col_cells:
            if col_letter is None:
                col_letter = get_column_letter(cell.column)
            try:
                value_len = len(str(cell.value)) if cell.value is not None else 0
            except Exception:  # noqa: BLE001
                value_len = 0
            length = max(length, value_len)
        if col_letter:
            ws.column_dimensions[col_letter].width = min(max(length + 2, min_width), max_width)


def _style_header_row(ws: Worksheet, row: int, n_cols: int) -> None:
    """Memberi style bold + fill pada baris header."""
    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=col_idx)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = CENTER_ALIGN
        cell.border = THIN_BORDER


def _write_dataframe(ws: Worksheet, df: pd.DataFrame, start_row: int = 1) -> int:
    """Menulis dataframe ke worksheet mulai dari start_row. Mengembalikan baris terakhir."""
    for col_idx, col_name in enumerate(df.columns, start=1):
        ws.cell(row=start_row, column=col_idx, value=str(col_name))
    _style_header_row(ws, start_row, len(df.columns))

    for row_offset, (_, row_data) in enumerate(df.iterrows(), start=1):
        for col_idx, value in enumerate(row_data, start=1):
            if isinstance(value, (pd.Timestamp,)):
                value = value.to_pydatetime()
            if pd.isna(value):
                value = None
            ws.cell(row=start_row + row_offset, column=col_idx, value=value)

    return start_row + len(df)


def write_pair_result_sheet(wb: Workbook, result: McNemarResult) -> None:
    """Sheet 1 — Pair Result: data asli + kolom *_correct."""
    ws = wb.create_sheet("Pair Result")
    last_row = _write_dataframe(ws, result.df, start_row=1)

    method1_col_name = f"{result.method1_name}_correct"
    method2_col_name = f"{result.method2_name}_correct"
    col_names = list(result.df.columns)

    ws.freeze_panes = "A2"
    _autofit_columns(ws)

    for col_name in (method1_col_name, method2_col_name):
        if col_name in col_names:
            col_idx = col_names.index(col_name) + 1
            col_letter = get_column_letter(col_idx)
            for r in range(2, last_row + 1):
                ws[f"{col_letter}{r}"].alignment = CENTER_ALIGN


def write_summary_sheet(wb: Workbook, result: McNemarResult) -> None:
    """Sheet 2 — McNemar Summary."""
    ws = wb.create_sheet("McNemar Summary")

    exact_label = "Exact (Binomial)" if result.is_exact else "Chi-square Approximation"

    rows: list[tuple[str, object]] = [
        ("Nama Method 1", result.method1_name),
        ("Nama Method 2", result.method2_name),
        ("Total Data", result.total_data),
        (f"Accuracy - {result.method1_name}", result.accuracy_method1),
        (f"Accuracy - {result.method2_name}", result.accuracy_method2),
        ("a (kedua metode benar)", result.a),
        ("b (Method1 benar, Method2 salah)", result.b),
        ("c (Method1 salah, Method2 benar)", result.c),
        ("d (kedua metode salah)", result.d),
        ("Jenis McNemar", exact_label),
        ("Continuity Correction", "Ya" if result.use_correction else "Tidak"),
        ("Statistic", result.statistic),
        ("p-value", result.p_value),
        ("Alpha", result.alpha),
        ("Decision", result.decision),
        ("Interpretation", result.interpretation),
    ]

    ws.cell(row=1, column=1, value="Ringkasan Hasil McNemar Test").font = TITLE_FONT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=2)

    accuracy_row_indices = []
    start_row = 3
    for offset, (label, value) in enumerate(rows):
        r = start_row + offset
        label_cell = ws.cell(row=r, column=1, value=label)
        label_cell.font = LABEL_FONT
        value_cell = ws.cell(row=r, column=2, value=value)
        if label.startswith("Accuracy"):
            value_cell.number_format = "0.00%"
            accuracy_row_indices.append(r)
        if label in ("p-value", "Alpha", "Statistic"):
            value_cell.number_format = "0.000000"
        label_cell.border = THIN_BORDER
        value_cell.border = THIN_BORDER

    ws.column_dimensions["A"].width = 38
    ws.column_dimensions["B"].width = 70


def write_contingency_sheet(wb: Workbook, result: McNemarResult) -> None:
    """Sheet 3 — Contingency Table, header memakai nama metode sebenarnya."""
    ws = wb.create_sheet("Contingency Table")

    m1 = result.method1_name
    m2 = result.method2_name

    ws.cell(row=1, column=1, value="Tabel Kontingensi McNemar").font = TITLE_FONT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=3)

    header_row = 3
    ws.cell(row=header_row, column=1, value="")
    ws.cell(row=header_row, column=2, value=f"{m2} Correct")
    ws.cell(row=header_row, column=3, value=f"{m2} Incorrect")
    _style_header_row(ws, header_row, 3)

    data_rows = [
        (f"{m1} Correct", result.a, result.b),
        (f"{m1} Incorrect", result.c, result.d),
    ]

    for offset, (row_label, val1, val2) in enumerate(data_rows, start=1):
        r = header_row + offset
        label_cell = ws.cell(row=r, column=1, value=row_label)
        label_cell.font = HEADER_FONT
        label_cell.fill = HEADER_FILL
        label_cell.alignment = CENTER_ALIGN
        label_cell.border = THIN_BORDER

        c2 = ws.cell(row=r, column=2, value=val1)
        c3 = ws.cell(row=r, column=3, value=val2)
        for c in (c2, c3):
            c.alignment = CENTER_ALIGN
            c.border = THIN_BORDER

    _autofit_columns(ws)


EXPLANATION_TEXT: list[tuple[str, str]] = [
    (
        "Tujuan McNemar Test",
        "McNemar Test digunakan untuk membandingkan performa dua metode "
        "klasifikasi pada data berpasangan (paired data) yang sama, guna "
        "menguji apakah terdapat perbedaan proporsi kesalahan/kebenaran "
        "yang signifikan secara statistik antara kedua metode tersebut.",
    ),
    (
        "Arti a, b, c, d",
        "a = jumlah data di mana kedua metode memberikan prediksi yang benar. "
        "b = jumlah data di mana Method 1 benar namun Method 2 salah. "
        "c = jumlah data di mana Method 1 salah namun Method 2 benar. "
        "d = jumlah data di mana kedua metode memberikan prediksi yang salah. "
        "McNemar Test hanya berfokus pada sel b dan c (kasus yang berbeda "
        "hasilnya antar metode), karena sel a dan d tidak memberikan "
        "informasi tentang perbedaan performa.",
    ),
    (
        "Arti Statistic",
        "Statistic adalah nilai uji (test statistic) yang dihasilkan dari "
        "McNemar Test. Untuk exact test, statistic mengikuti distribusi "
        "binomial dari sel b dan c. Untuk chi-square approximation, "
        "statistic mengikuti distribusi chi-square dengan 1 derajat bebas.",
    ),
    (
        "Arti p-value",
        "p-value adalah probabilitas memperoleh hasil sekstrem atau lebih "
        "ekstrem dari hasil observasi, dengan asumsi H0 (tidak ada "
        "perbedaan performa antar metode) benar. Semakin kecil p-value, "
        "semakin kuat bukti untuk menolak H0.",
    ),
    (
        "Kapan menggunakan Exact Test",
        "Exact Test (binomial) digunakan ketika jumlah (b + c) kecil, yaitu "
        "kurang dari 25. Pada kondisi ini, pendekatan chi-square kurang "
        "akurat karena asumsi distribusi normal tidak terpenuhi dengan baik.",
    ),
    (
        "Kapan menggunakan Chi-square Approximation",
        "Chi-square Approximation (dengan continuity correction) digunakan "
        "ketika jumlah (b + c) cukup besar, yaitu 25 atau lebih, sehingga "
        "distribusi chi-square dapat mendekati distribusi sebenarnya "
        "dengan baik.",
    ),
    (
        "Arti Reject H0",
        "Reject H0 berarti terdapat cukup bukti statistik (p-value < alpha) "
        "untuk menyimpulkan bahwa kedua metode memiliki performa yang "
        "berbeda secara signifikan.",
    ),
    (
        "Arti Fail to Reject H0",
        "Fail to Reject H0 berarti tidak terdapat cukup bukti statistik "
        "(p-value >= alpha) untuk menyimpulkan adanya perbedaan performa "
        "antara kedua metode. Ini bukan berarti kedua metode identik, "
        "hanya berarti data yang ada belum cukup membuktikan adanya "
        "perbedaan.",
    ),
    (
        "Arti Accuracy",
        "Accuracy adalah proporsi prediksi yang benar dibandingkan dengan "
        "seluruh data, dihitung sebagai (jumlah prediksi benar) / (total "
        "data). Accuracy memberikan gambaran umum performa masing-masing "
        "metode secara individual, namun tidak menunjukkan signifikansi "
        "statistik perbedaan antar metode - untuk itu digunakan McNemar Test.",
    ),
    (
        "Interpretasi Hasil",
        "Interpretasi hasil McNemar Test dilakukan dengan membandingkan "
        "p-value terhadap alpha (umumnya 0.05). Jika p-value lebih kecil "
        "dari alpha, maka kedua metode dianggap memiliki performa yang "
        "berbeda secara signifikan. Jika sebaliknya, maka tidak terdapat "
        "cukup bukti untuk menyatakan adanya perbedaan performa.",
    ),
    (
        "Contoh Interpretasi (p < 0.05)",
        "p < 0.05 menunjukkan kedua metode memiliki performa yang berbeda "
        "secara signifikan.",
    ),
    (
        "Contoh Interpretasi (p >= 0.05)",
        "p >= 0.05 menunjukkan tidak terdapat perbedaan performa yang "
        "signifikan.",
    ),
]


def write_explanation_sheet(wb: Workbook) -> None:
    """Sheet 4 — Explanation."""
    ws = wb.create_sheet("Explanation")

    ws.cell(row=1, column=1, value="Penjelasan McNemar Test").font = TITLE_FONT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=2)

    header_row = 3
    ws.cell(row=header_row, column=1, value="Topik")
    ws.cell(row=header_row, column=2, value="Penjelasan")
    _style_header_row(ws, header_row, 2)

    for offset, (topic, explanation) in enumerate(EXPLANATION_TEXT, start=1):
        r = header_row + offset
        topic_cell = ws.cell(row=r, column=1, value=topic)
        topic_cell.font = LABEL_FONT
        topic_cell.alignment = Alignment(vertical="top", wrap_text=True)
        topic_cell.border = THIN_BORDER

        exp_cell = ws.cell(row=r, column=2, value=explanation)
        exp_cell.alignment = Alignment(vertical="top", wrap_text=True)
        exp_cell.border = THIN_BORDER

    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 100


def write_metadata_sheet(
    wb: Workbook,
    input_file: Path,
    output_file: Path,
    sheet_source: str,
    result: McNemarResult,
) -> None:
    """Sheet 5 — Metadata."""
    ws = wb.create_sheet("Metadata")

    ws.cell(row=1, column=1, value="Metadata Analisis").font = TITLE_FONT
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=2)

    rows: list[tuple[str, object]] = [
        ("Input file", str(input_file)),
        ("Output file", str(output_file)),
        ("Sheet source", sheet_source),
        ("Nama Method 1", result.method1_name),
        ("Nama Method 2", result.method2_name),
        ("Total data", result.total_data),
        ("Analysis date", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
        ("Python version", sys.version.split()[0]),
        ("Pandas version", pd.__version__),
        ("Statsmodels version", statsmodels.__version__),
        ("Generated automatically", "Ya, seluruh isi file ini dihasilkan secara otomatis oleh program."),
    ]

    start_row = 3
    for offset, (label, value) in enumerate(rows, start=0):
        r = start_row + offset
        label_cell = ws.cell(row=r, column=1, value=label)
        label_cell.font = LABEL_FONT
        label_cell.border = THIN_BORDER
        value_cell = ws.cell(row=r, column=2, value=value)
        value_cell.border = THIN_BORDER
        value_cell.alignment = Alignment(wrap_text=True, vertical="top")

    ws.column_dimensions["A"].width = 26
    ws.column_dimensions["B"].width = 80

def write_effect_size_sheet(
    wb: Workbook,
    result: McNemarResult,
) -> None:

    ws = wb.create_sheet("Effect Size")
    ws["A1"] = "Effect Size (Matched-pairs Odds Ratio)"
    ws["A1"].font = TITLE_FONT
    rows = [
        ("Method 1", result.method1_name),
        ("Method 2", result.method2_name),
        ("b (Method1 benar, Method2 salah)", result.b),
        ("c (Method1 salah, Method2 benar)", result.c),
        ("Odds Ratio", result.effect_size.odds_ratio),
        ("Log Odds Ratio", result.effect_size.log_odds_ratio),
        ("Standard Error", result.effect_size.standard_error),
        ("95% CI Lower", result.effect_size.ci_lower),
        ("95% CI Upper", result.effect_size.ci_upper),
        ("Direction", result.effect_size.direction),
        ("Magnitude", result.effect_size.magnitude),
        ("Interpretation", result.effect_size.interpretation),
    ]
    start_row = 3
    for i, (label, value) in enumerate(rows):
        r = start_row + i
        c1 = ws.cell(r, 1, label)
        c2 = ws.cell(r, 2, value)
        c1.font = LABEL_FONT
        c1.border = THIN_BORDER
        c2.border = THIN_BORDER
        if isinstance(value, float):
            c2.number_format = "0.0000"

    ws.column_dimensions["A"].width = 40
    ws.column_dimensions["B"].width = 60

def save_results_to_excel(
    result: McNemarResult,
    input_file: Path,
    output_file: Path,
    sheet_source: str = SHEET_SOURCE_NAME,
) -> None:
    """Menyusun seluruh sheet output dan menyimpan ke file Excel."""
    wb = Workbook()
    # Hapus sheet default kosong yang otomatis dibuat.
    default_sheet = wb.active
    wb.remove(default_sheet)

    write_pair_result_sheet(wb, result)
    write_summary_sheet(wb, result)
    write_contingency_sheet(wb, result)
    write_explanation_sheet(wb)
    write_metadata_sheet(wb, input_file, output_file, sheet_source, result)
    write_effect_size_sheet(wb, result)

    output_file.parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_file)


# --------------------------------------------------------------------------- #
# 6. Fungsi utama (end-to-end), khusus dipanggil dari dalam Jupyter Notebook
# --------------------------------------------------------------------------- #

def build_output_path(input_file: Path, output_file: Optional[Path] = None) -> Path:
    """Membentuk nama file output default: <nama_file>_mcnemar.xlsx"""
    if output_file is not None:
        return output_file
    return input_file.with_name(f"{input_file.stem}_mcnemar.xlsx")


def run_mcnemar_analysis(
    input_path: str | Path,
    sheet_name: str = SHEET_SOURCE_NAME,
    alpha: float = 0.05,
    output_path: Optional[str | Path] = None,
) -> Path:
    """
    Fungsi end-to-end: membaca file input, menjalankan analisis McNemar,
    dan menyimpan hasil ke file Excel output.

    Returns:
        Path ke file Excel output yang dihasilkan.
    """
    input_file = Path(input_path)
    output_file = build_output_path(input_file, Path(output_path) if output_path else None)

    result = analyze_mcnemar(input_file, sheet_name=sheet_name, alpha=alpha)
    save_results_to_excel(result, input_file, output_file, sheet_source=sheet_name)

    return output_file


def print_result_summary(result: McNemarResult) -> None:
    """Mencetak ringkasan hasil McNemar Test dengan rapi (cocok untuk notebook)."""
    exact_label = "Exact (Binomial)" if result.is_exact else "Chi-square Approximation"
    line = "=" * 70
    print(line)
    print("RINGKASAN HASIL McNEMAR TEST")
    print(line)
    print(f"Method 1               : {result.method1_name}")
    print(f"Method 2               : {result.method2_name}")
    print(f"Total Data             : {result.total_data}")
    print(f"Accuracy {result.method1_name:<15}: {result.accuracy_method1:.2%}")
    print(f"Accuracy {result.method2_name:<15}: {result.accuracy_method2:.2%}")
    print("-" * 70)
    print(f"a (kedua benar)        : {result.a}")
    print(f"b (M1 benar, M2 salah) : {result.b}")
    print(f"c (M1 salah, M2 benar) : {result.c}")
    print(f"d (kedua salah)        : {result.d}")
    print("-" * 70)
    print(f"Jenis McNemar          : {exact_label}")
    print(f"Continuity Correction  : {'Ya' if result.use_correction else 'Tidak'}")
    print(f"Statistic              : {result.statistic:.6f}")
    print(f"p-value                : {result.p_value:.6f}")
    print(f"Alpha                  : {result.alpha}")
    print(f"Decision               : {result.decision}")
    print("-" * 70)
    print("EFFECT SIZE")
    print("-" * 70)
    print(f"Odds Ratio            : {result.effect_size.odds_ratio:.4f}")
    print(f"Log Odds Ratio        : {result.effect_size.log_odds_ratio:.4f}")
    print(f"Standard Error        : {result.effect_size.standard_error:.4f}")
    print(
        f"95% Confidence Interval : "
        f"[{result.effect_size.ci_lower:.4f}, "
        f"{result.effect_size.ci_upper:.4f}]"
    )
    print(f"Direction             : {result.effect_size.direction}")
    print(f"Magnitude             : {result.effect_size.magnitude}")
    print(f"Interpretation        :")
    print(f"  {result.effect_size.interpretation}")
    print(line)
    print(result.interpretation)
    print(line)


def run_mcnemar_notebook(
    input_path: str | Path,
    sheet_name: str = SHEET_SOURCE_NAME,
    alpha: float = 0.05,
    output_path: Optional[str | Path] = None,
    verbose: bool = True,
) -> tuple[McNemarResult, Path]:
    """
    Versi ramah-Jupyter dari `run_mcnemar_analysis`.

    Berbeda dengan `run_mcnemar_analysis` (yang hanya mengembalikan path
    output), fungsi ini mengembalikan objek `McNemarResult` sekaligus path
    file output, sehingga hasilnya (termasuk `result.df`) dapat langsung
    diinspeksi lebih lanjut di dalam notebook, dan otomatis mencetak
    ringkasan hasil bila `verbose=True`.

    Contoh:
        from mcnemar_test import run_mcnemar_notebook

        result, output_path = run_mcnemar_notebook("data.xlsx")
        result.df.head()          # DataFrame dengan kolom *_correct
        result.p_value            # akses langsung nilai p-value
        print(output_path)        # path file Excel hasil analisis

    Raises:
        McNemarAnalysisError (dan turunannya) jika terjadi error validasi.
        Error ditangkap secara eksplisit (bukan silent) agar tetap terlihat
        jelas sebagai traceback di dalam cell notebook.
    """
    input_file = Path(input_path)
    output_file = build_output_path(input_file, Path(output_path) if output_path else None)

    result = analyze_mcnemar(input_file, sheet_name=sheet_name, alpha=alpha)
    save_results_to_excel(result, input_file, output_file, sheet_source=sheet_name)

    if verbose:
        print_result_summary(result)
        print(f"\nFile hasil disimpan di: '{output_file}'")

    return result, output_file


def analyze(
    input_path: str | Path,
    sheet_name: str = SHEET_SOURCE_NAME,
    alpha: float = 0.05,
    output_path: Optional[str | Path] = None,
    verbose: bool = True,
) -> tuple[McNemarResult, Path]:
    """
    Alias singkat untuk `run_mcnemar_notebook`, dipakai sebagai satu-satunya
    pintu masuk (entry point) yang direkomendasikan saat bekerja di Jupyter
    Notebook / Google Colab.

    Contoh pemakaian di dalam cell notebook:

        from mcnemar_test import analyze

        result, output_path = analyze("data.xlsx")

        result.p_value
        result.decision
        result.df.head()

    Jika terjadi error (file tidak ditemukan, sheet tidak ada, kolom tidak
    lengkap, dsb.), exception akan langsung tampil sebagai traceback di
    dalam cell — ini disengaja, karena traceback jauh lebih mudah dibaca
    dan ditelusuri di notebook dibanding pesan error yang diredam.
    """
    return run_mcnemar_notebook(
        input_path=input_path,
        sheet_name=sheet_name,
        alpha=alpha,
        output_path=output_path,
        verbose=verbose,
    )

In [5]:
print("-" * 70)
print("McNemar Test Analysis AST vs AST+Graph2vec (PRAKTIKUM)")
result, output_path = analyze(
    "output(2)/eval/praktikum/praktikum_comparison_AST_ASTGraph2Vec.xlsx",
    sheet_name="Detail Comparison",  # default
    alpha=0.05,                       # default
    output_path="output(2)/eval/praktikum/praktikum_comparison_AST_ASTGraph2Vec_mcnemar.xlsx",                 # default -> <nama_file>_mcnemar.xlsx
    verbose=True,                     # default -> cetak ringkasan
)

----------------------------------------------------------------------
McNemar Test Analysis AST vs AST+Graph2vec (PRAKTIKUM)
RINGKASAN HASIL McNEMAR TEST
Method 1               : AST
Method 2               : ASTGraph2Vec
Total Data             : 4024
Accuracy AST            : 63.79%
Accuracy ASTGraph2Vec   : 77.93%
----------------------------------------------------------------------
a (kedua benar)        : 2303
b (M1 benar, M2 salah) : 264
c (M1 salah, M2 benar) : 833
d (kedua salah)        : 624
----------------------------------------------------------------------
Jenis McNemar          : Chi-square Approximation
Continuity Correction  : Ya
Statistic              : 294.096627
p-value                : 0.000000
Alpha                  : 0.05
Decision               : Reject H0
----------------------------------------------------------------------
EFFECT SIZE
----------------------------------------------------------------------
Odds Ratio            : 3.1553
Log Odds Ratio        : 1

In [6]:
print("-" * 70)
print("McNemar Test Analysis AST vs AST+Graph2vec (TUGAS)")
result, output_path = analyze(
    "output(2)/eval/tugas/tugas_comparison_AST_ASTGraph2Vec.xlsx",
    sheet_name="Detail Comparison",  # default
    alpha=0.05,                       # default
    output_path="output(2)/eval/tugas/tugas_comparison_AST_ASTGraph2Vec_mcnemar.xlsx",                 # default -> <nama_file>_mcnemar.xlsx
    verbose=True,                     # default -> cetak ringkasan
)

----------------------------------------------------------------------
McNemar Test Analysis AST vs AST+Graph2vec (TUGAS)
RINGKASAN HASIL McNEMAR TEST
Method 1               : AST
Method 2               : ASTGraph2Vec
Total Data             : 1360
Accuracy AST            : 69.71%
Accuracy ASTGraph2Vec   : 81.62%
----------------------------------------------------------------------
a (kedua benar)        : 886
b (M1 benar, M2 salah) : 62
c (M1 salah, M2 benar) : 224
d (kedua salah)        : 188
----------------------------------------------------------------------
Jenis McNemar          : Chi-square Approximation
Continuity Correction  : Ya
Statistic              : 90.632867
p-value                : 0.000000
Alpha                  : 0.05
Decision               : Reject H0
----------------------------------------------------------------------
EFFECT SIZE
----------------------------------------------------------------------
Odds Ratio            : 3.6129
Log Odds Ratio        : 1.2845
S

In [7]:
print("-" * 70)
print("McNemar Test Analysis JPLAG vs AST+Graph2vec (PRAKTIKUM)")
result, output_path = analyze(
    "output(2)/eval/praktikum/praktikum_comparison_JPLAG_ASTGraph2Vec.xlsx",
    sheet_name="Detail Comparison",  # default
    alpha=0.05,                       # default
    output_path="output(2)/eval/praktikum/praktikum_comparison_JPLAG_ASTGraph2Vec_mcnemar.xlsx",                 # default -> <nama_file>_mcnemar.xlsx
    verbose=True,                     # default -> cetak ringkasan
)

----------------------------------------------------------------------
McNemar Test Analysis JPLAG vs AST+Graph2vec (PRAKTIKUM)
RINGKASAN HASIL McNEMAR TEST
Method 1               : JPLAG
Method 2               : ASTGraph2Vec
Total Data             : 4024
Accuracy JPLAG          : 76.17%
Accuracy ASTGraph2Vec   : 77.93%
----------------------------------------------------------------------
a (kedua benar)        : 3037
b (M1 benar, M2 salah) : 28
c (M1 salah, M2 benar) : 99
d (kedua salah)        : 860
----------------------------------------------------------------------
Jenis McNemar          : Chi-square Approximation
Continuity Correction  : Ya
Statistic              : 38.582677
p-value                : 0.000000
Alpha                  : 0.05
Decision               : Reject H0
----------------------------------------------------------------------
EFFECT SIZE
----------------------------------------------------------------------
Odds Ratio            : 3.5357
Log Odds Ratio        : 

In [8]:
print("-" * 70)
print("McNemar Test Analysis JPLAG vs AST+Graph2vec (TUGAS)")
result, output_path = analyze(
    "output(2)/eval/tugas/tugas_comparison_JPLAG_ASTGraph2Vec.xlsx",
    sheet_name="Detail Comparison",  # default
    alpha=0.05,                       # default
    output_path="output(2)/eval/tugas/tugas_comparison_JPLAG_ASTGraph2Vec_mcnemar.xlsx",                 # default -> <nama_file>_mcnemar.xlsx
    verbose=True,                     # default -> cetak ringkasan
)

----------------------------------------------------------------------
McNemar Test Analysis JPLAG vs AST+Graph2vec (TUGAS)
RINGKASAN HASIL McNEMAR TEST
Method 1               : JPLAG
Method 2               : ASTGraph2Vec
Total Data             : 1360
Accuracy JPLAG          : 75.88%
Accuracy ASTGraph2Vec   : 81.62%
----------------------------------------------------------------------
a (kedua benar)        : 1027
b (M1 benar, M2 salah) : 5
c (M1 salah, M2 benar) : 83
d (kedua salah)        : 245
----------------------------------------------------------------------
Jenis McNemar          : Chi-square Approximation
Continuity Correction  : Ya
Statistic              : 67.375000
p-value                : 0.000000
Alpha                  : 0.05
Decision               : Reject H0
----------------------------------------------------------------------
EFFECT SIZE
----------------------------------------------------------------------
Odds Ratio            : 16.6000
Log Odds Ratio        : 2.80